First, PyTorch is a tensor library that extends the concept of the array-oriented pro-
gramming library NumPy with the additional feature that accelerates computation on
GPUs, thus providing a seamless switch between CPUs and GPUs. 

Second, PyTorch is an automatic differentiation engine, also known as autograd, that enables the automatic
computation of gradients for tensor operations, simplifying backpropagation and
model optimization. 

Finally, PyTorch is a deep learning library. It offers modular, flexi-
ble, and efficient building blocks, including pretrained models, loss functions, and
optimizers, for designing and training a wide range of deep learning models, catering
to both researchers and developers.


In [1]:
import torch

In [2]:
torch.__version__

'2.10.0'

In [3]:
torch.cuda.is_available()

False

In [4]:
print(torch.backends.mps.is_available())

True


In [6]:
tensor0d = torch.tensor(3.14)
tensor1d = torch.tensor([1, 2, 3])
tensor2d = torch.tensor([[1, 2, 3], [4, 5, 6]])
tensor3d = torch.tensor([[[1, 2, 3], [4, 5, 6]], [[7, 8, 9], [10, 11, 12]]])

In [8]:
tensor3d.dtype

torch.int64

In [9]:
floatvec = torch.tensor([1.0, 2.0, 3.0])
print(floatvec.dtype)

torch.float32


In [12]:
print(tensor3d)

tensor([[[ 1,  2,  3],
         [ 4,  5,  6]],

        [[ 7,  8,  9],
         [10, 11, 12]]])


In [15]:
tensor3d.shape

torch.Size([2, 2, 3])

In [ ]:
tensor3d.reshape(3, 2,2)

tensor([[[ 1,  2],
         [ 3,  4]],

        [[ 5,  6],
         [ 7,  8]],

        [[ 9, 10],
         [11, 12]]])

In [18]:
tensor3d.T

tensor([[[ 1,  7],
         [ 4, 10]],

        [[ 2,  8],
         [ 5, 11]],

        [[ 3,  9],
         [ 6, 12]]])

In [19]:
tensor2d

tensor([[1, 2, 3],
        [4, 5, 6]])

In [20]:
tensor2d.T

tensor([[1, 4],
        [2, 5],
        [3, 6]])

In [21]:
tensor2d.matmul(tensor2d.T)

tensor([[14, 32],
        [32, 77]])

In [22]:
tensor2d@tensor2d.T

tensor([[14, 32],
        [32, 77]])

A.3 A.3 Seeing models as computation graphs

A computational graph is a directed graph that allows us to express and visualize
mathematical expressions.

Listing A.2 A logistic regression forward pass

In [23]:
import torch.nn.functional as F

In [24]:
y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2])
b = torch.tensor([0.0])
z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)

In [29]:
print(z)
print(a)
print(loss)

tensor([2.4200])
tensor([0.9183])
tensor(0.0852)


In [32]:
import torch.nn.functional as F
from torch.autograd import grad
y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)
z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)
grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

In [33]:
print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [34]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


In [ ]:
torch.nn.Module

Listing A.4 A multilayer perceptron with two hidden layers

In [35]:
class NeuralNetwork(torch.nn.Module):

    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        self.layers = torch.nn.Sequential(
        # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),
            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),
            # output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

In [36]:
model = NeuralNetwork(50, 3)

In [37]:
print(model)


NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [38]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of trainable model parameters:", num_params)

Total number of trainable model parameters: 2213


In [39]:
print(model.layers[0].weight)

Parameter containing:
tensor([[ 0.1405, -0.0804,  0.1302,  ..., -0.0043, -0.1220, -0.0632],
        [-0.0792,  0.1180, -0.1115,  ..., -0.0077,  0.0527,  0.0043],
        [ 0.0383,  0.1253,  0.0708,  ..., -0.1370, -0.0086, -0.0251],
        ...,
        [ 0.0709, -0.0954, -0.0796,  ..., -0.0473,  0.1356,  0.0243],
        [-0.0678, -0.0131,  0.0547,  ...,  0.0088, -0.1239,  0.1346],
        [-0.0935,  0.0444,  0.1184,  ...,  0.0944,  0.0470,  0.0774]],
       requires_grad=True)


In [40]:
print(model.layers[0].weight.shape)

torch.Size([30, 50])


In [41]:
model.layers[0].bias

Parameter containing:
tensor([ 0.1268, -0.1405, -0.1207, -0.0016,  0.0770,  0.0311,  0.0542,  0.1195,
        -0.0132, -0.1363, -0.1131,  0.1137,  0.0540, -0.0427, -0.0163, -0.1332,
         0.0871,  0.0859, -0.0546, -0.0589, -0.0918, -0.0039, -0.0125,  0.1201,
         0.0934, -0.0577,  0.0921,  0.0483,  0.0059,  0.0367],
       requires_grad=True)

In [42]:
torch.manual_seed(123)
model = NeuralNetwork(50, 3)
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


In [43]:
torch.manual_seed(123)
X = torch.rand((1, 50))
out = model(X)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


In [44]:
with torch.no_grad():
    out = model(X)
    print(out)

tensor([[-0.1262,  0.1080, -0.1792]])


In [45]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
    print(out)

tensor([[0.3113, 0.3934, 0.2952]])


Listing A.5 Creating a small toy dataset

In [46]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])
y_test = torch.tensor([0, 1])

Listing A.6 Defining a custom Dataset class

In [47]:
from torch.utils.data import Dataset
class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y
    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y
    def __len__(self):
        return self.labels.shape[0]
train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)